### RAG Pipelines- Data Ingestion to Vector Database Pipeline 

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Admin\AppData\Local\Temp\ipykernel_21192\1659607159.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
c:\Users\Admin\OneDrive\Desktop\PROJECTS\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

In [3]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data") 

Found 7 PDF files to process.

Processing file: ..\data\pdf\A better way to teach children to read in India.pdf
Loaded 42 pages

Processing file: ..\data\pdf\Attention.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/GKSXYT+TimesNewRomanPS-BoldItalicMT', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 99, '/FontDescriptor': IndirectObject(28, 0, 2158779080592), '/LastChar': 116, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [444, 0, 444, 0, 0, 0, 278, 0, 0, 278, 0, 556, 500, 0, 0, 389, 389, 278]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/MCBCCI+SourceSansRoman-Regular', '/Encoding': IndirectObject(50, 0, 2158779080592), '/FirstChar': 1, '/FontDescriptor': IndirectObject(51, 0, 2158779080592), '/LastChar': 8, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [456, 544, 496, 495, 200, 292, 542, 347]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a C

Loaded 44 pages

Processing file: ..\data\pdf\CNN.pdf
Loaded 28 pages

Processing file: ..\data\pdf\EC4102_MidTerm_Proposal.pdf
Loaded 5 pages

Processing file: ..\data\pdf\Mid_term_applied_econometrics_group_6.pdf
Loaded 5 pages

Processing file: ..\data\pdf\SadakVision — INDIS 2026.pdf
Loaded 16 pages

Processing file: ..\data\pdf\SV_IEEE.pdf
Loaded 8 pages

Total documents loaded: 148


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 9.2.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-01-19T12:19:10-05:00', 'author': 'Leigh', 'moddate': '2010-01-19T12:36:55-05:00', 'title': 'Microsoft Word - Shishu Working Paper 2009-06-1', 'source': '..\\data\\pdf\\A better way to teach children to read in India.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': 'A better way to teach children to read in India.pdf', 'file_type': 'pdf'}, page_content="A Better Way to Teach Children to Read? \nEvidence from a Randomized Controlled Trial1 \n \n \n \n \nMay 2009 \n \nFang He \n(Columbia University) \n \nLeigh L. Linden2 \n(Columbia University, BREAD, and IZA) \n \nMargaret MacLeod \n(Columbia University and U.S. Department of State) \n \n \n \n \n \n \n \n \nAbstract: Using\xa0three\xa0randomized\xa0evaluations,\xa0we\xa0assess\xa0the\xa0relative\xa0efficacy\xa0of\xa0a\xa0literacy\xa0skills\xa0\ndevelopment\xa0program\xa0in\xa0Mumbai,\xa

### Text Splitting and making Chunks

In [5]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata} ")
    return split_docs
   

In [6]:
chunks = split_documents(all_pdf_documents)

Split 148 documents into 440 chunks



Example chunk:
Content: A Better Way to Teach Children to Read? 
Evidence from a Randomized Controlled Trial1 
 
 
 
 
May 2009 
 
Fang He 
(Columbia University) 
 
Leigh L. Linden2 
(Columbia University, BREAD, and IZA) 
 
...
Metadata: {'producer': 'Acrobat Distiller 9.2.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-01-19T12:19:10-05:00', 'author': 'Leigh', 'moddate': '2010-01-19T12:36:55-05:00', 'title': 'Microsoft Word - Shishu Working Paper 2009-06-1', 'source': '..\\data\\pdf\\A better way to teach children to read in India.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': 'A better way to teach children to read in India.pdf', 'file_type': 'pdf'} 


### Embedding and Vector Store DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
    ):
        """Initialize the EmbeddingManager."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self) -> None:
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully: {self.model_name}. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [9]:
# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2607.44it/s]


Model loaded successfully: all-MiniLM-L6-v2. Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_21192\3387099033.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


### VectorStore

In [10]:
import hashlib


class VectorStore:
    """Handles storage of document embeddings using ChromaDB."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persistent_directory: str = "../data/vector_store"
    ):
        """
        Initialize the VectorStore.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persistent_directory (str): Directory where the vector
                store will be persisted.
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self) -> None:
        """Initialize the persistent ChromaDB vector store."""

        try:
            # Create the directory if it doesn't exist
            os.makedirs(
                self.persistent_directory,
                exist_ok=True
            )

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            # Get existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def reset_collection(self) -> None:
        """Delete the collection and re-create it empty."""
        try:
            self.client.delete_collection(name=self.collection_name)
        except Exception:
            pass  # collection didn't exist yet

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings for RAG"
            }
        )
        print(f"Collection '{self.collection_name}' reset (now empty).")

    @staticmethod
    def _make_id(doc) -> str:
        """Deterministic ID: same source + page + text always gives the same ID."""
        key = (
            f"{doc.metadata.get('source_file', doc.metadata.get('source', ''))}"
            f"|{doc.metadata.get('page', '')}"
            f"|{doc.page_content}"
        )
        return "doc_" + hashlib.sha256(key.encode("utf-8")).hexdigest()[:32]

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray,
        replace: bool = True
    ) -> None:
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Corresponding embeddings for the documents.
            replace: If True (default), wipe the collection first so the
                stored data exactly matches the documents passed in.
                Re-running ingestion never creates duplicates.
        """

        if not documents or embeddings.size == 0:
            raise ValueError(
                "Documents and embeddings cannot be empty"
            )

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must match"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to the vector store..."
        )

        if replace:
            self.reset_collection()

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Deterministic document ID (content hash)
            doc_id = self._make_id(doc)
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Store original document text
            documents_text.append(
                doc.page_content
            )

            # Convert NumPy array to Python list
            embeddings_list.append(
                embedding.tolist()
            )

        # Add everything to ChromaDB
        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to the vector store."
            )

            print(
                f"Total documents in collection after addition: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise

In [11]:
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 440


In [12]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 9.2.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-01-19T12:19:10-05:00', 'author': 'Leigh', 'moddate': '2010-01-19T12:36:55-05:00', 'title': 'Microsoft Word - Shishu Working Paper 2009-06-1', 'source': '..\\data\\pdf\\A better way to teach children to read in India.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': 'A better way to teach children to read in India.pdf', 'file_type': 'pdf'}, page_content='A Better Way to Teach Children to Read? \nEvidence from a Randomized Controlled Trial1 \n \n \n \n \nMay 2009 \n \nFang He \n(Columbia University) \n \nLeigh L. Linden2 \n(Columbia University, BREAD, and IZA) \n \nMargaret MacLeod \n(Columbia University and U.S. Department of State) \n \n \n \n \n \n \n \n \nAbstract: Using\xa0three\xa0randomized\xa0evaluations,\xa0we\xa0assess\xa0the\xa0relative\xa0efficacy\xa0of\xa0a\xa0literacy\xa0skills\xa0\ndevelopment\xa0program\xa0in\xa0Mumbai,\xa

In [13]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 440 texts...


Batches: 100%|██████████| 14/14 [00:24<00:00,  1.73s/it]


Generated embeddings with shape: (440, 384)
Adding 440 documents to the vector store...
Collection 'pdf_documents' reset (now empty).
Successfully added 440 documents to the vector store.
Total documents in collection after addition: 440


### RAG Retriever from VectorStore (use query to retrieve relevant documents)

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score required.

        Returns:
            List of dictionaries containing:
            - document ID
            - document content
            - metadata
            - similarity score
            - distance
            - rank
        """

        print(f"Retrieving documents for query: '{query}'")
        print(
            f"Top K: {top_k}, "
            f"Score threshold: {score_threshold}"
        )

        # --------------------------------------------------
        # 1. Generate embedding for the user's query
        # --------------------------------------------------

        query_embedding = (
            self.embedding_manager
            .generate_embeddings([query])[0]
        )

        # --------------------------------------------------
        # 2. Search the vector store
        # --------------------------------------------------

        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            # --------------------------------------------------
            # 3. Process the search results
            # --------------------------------------------------

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                # --------------------------------------------------
                # 4. Process each retrieved document
                # --------------------------------------------------

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # ChromaDB returns cosine distance.
                    # Convert distance to similarity score.
                    similarity_score = 1 - distance

                    # --------------------------------------------------
                    # 5. Apply similarity threshold
                    # --------------------------------------------------

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:

                print("No documents found")

            return retrieved_docs

        except Exception as e:

            print(f"Error during retrieval: {e}")

            return []

In [15]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [16]:
rag_retriever.retrieve("What is sadak vision?")

Retrieving documents for query: 'What is sadak vision?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_d9a71857941dca31d2541cfcccbfe001',
  'content': 'Vision-level accuracy, 89% less compute , plus severity\nSensor only Vision only Fusion (SadakVision)\n0.48\n0.782 0.782 0.823 0.787 0.787\n0.606\n0.785 0.785\nPrecision Recall F1\nSame F1\nas vision-only, by design: the sensor decides what\nthe camera looks at, vision gives the verdict.\n 89%\nframes sent to the detector: 4,620 vs 43,200\nan hour.\n+ Severity\nlow / medium / high from the sensor trace.\nChecked so far on 62 human-labelled events.\nSadakVision · INDIS 2026 10',
  'metadata': {'total_pages': 16,
   'content_length': 464,
   'doc_index': 400,
   'page_label': '10',
   'source': '..\\data\\pdf\\SadakVision — INDIS 2026.pdf',
   'creationdate': '',
   'page': 9,
   'creator': 'PyPDF',
   'producer': 'appifact kit',
   'file_type': 'pdf',
   'title': 'SadakVision   INDIS 2026',
   'source_file': 'SadakVision — INDIS 2026.pdf'},
  'similarity_score': 0.09582984447479248,
  'distance': 0.9041701555252075,
  'rank':

### Intergration VectorDB Context pipeline with LLM output

In [17]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv


# Load environment variables from .env
load_dotenv()


# Get Groq API key
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")


# Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    # model_name="gemma2-9b-it",
    model_name = "openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


# Simple RAG pipeline:
# Retrieve relevant context + generate answer
def rag_simple(query, retriever, llm, top_k=3):

    # 1. Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    # 2. Combine retrieved document chunks into one context
    context = "\n\n".join(
        doc["content"]
        for doc in results
    ) if results else ""

    # 3. Handle case where no relevant documents are found
    if not context:
        print("No relevant documents found for the query.")
        return "No relevant documents found."

    # 4. Create the prompt for the LLM
    prompt = f"""
Use the following context to answer the question concisely.
Only use information from the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    # 5. Send the prompt to the Groq LLM
    response = llm.invoke(prompt)

    # 6. Return the generated answer
    return response.content




In [18]:
# Example usage
query = "What is the Attention Mechanism?"

answer = rag_simple(
    query=query,
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)

print("\nFinal Answer:")
print(answer)

Retrieving documents for query: 'What is the Attention Mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)



Final Answer:
**Answer:**  
An attention mechanism is a differentiable function that takes queries, keys, and values as input and produces an output that aggregates information from the values weighted by the similarity between queries and keys.


### Enhanced RAG Pipeline Features

In [19]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get(
            'source_file',
            doc['metadata'].get('source', 'unknown')
        ),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]

    confidence = max(
        [doc['similarity_score'] for doc in results]
    )

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(
        [prompt.format(context=context, query=query)]
    )

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context

    return output

In [20]:
# Example usage of the advanced RAG pipeline
result = rag_advanced(
    query="what is better way to teach attention mechanism  ?",
    retriever=rag_retriever,
    llm=llm,
    top_k=5,
    min_score=0.005,
    return_context=True
)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence Score:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is better way to teach attention mechanism  ?'
Top K: 5, Score threshold: 0.005
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **A concise recipe for “teaching” attention:**

1. **Pre‑train on a large, diverse corpus** – let the model discover useful patterns (positional, syntactic, broadcast) automatically.  
2. **Add explicit supervision** – when possible, provide attention‑target signals (e.g., alignments, parse trees, or saliency maps) and add a small attention‑loss term to the training objective.  
3. **Use a curriculum** – start with simple, local patterns (next/previous tokens, local spatial windows) and gradually introduce global, multi‑head patterns.  
4. **Combine channel‑ and spatial‑attention modules** – train them jointly so the network learns to aggregate information both across feature maps and across positions.  
5. **Fine‑tune on the target task** – keep the learned attention weights as a prior and let the model adapt them to the specific domain.  

This blend of unsupervised pre‑training, targeted supervision, curriculum learning, and joint channel/spatial attention yields the most ef

In [21]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""

        else:
            context = "\n\n".join(
                [doc["content"] for doc in results]
            )

            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get(
                        "page",
                        "unknown"
                    ),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "..."
                }
                for doc in results
            ]

            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {question}

Answer:"""

            if stream:
                print("Streaming answer:")

                for i in range(0, len(prompt), 80):
                    print(
                        prompt[i:i + 80],
                        end="",
                        flush=True
                    )
                    time.sleep(0.05)

                print()

            response = self.llm.invoke(
                [prompt.format(
                    context=context,
                    question=question
                )]
            )

            answer = response.content

        # Add citations to answer
        citations = [
            f"[{i + 1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = (
            answer
            + "\n\nCitations:\n"
            + "\n".join(citations)
            if citations
            else answer
        )

        # Optionally summarize answer
        summary = None

        if summarize and answer:
            summary_prompt = (
                f"Summarize the following answer in 2 sentences:\n"
                f"{answer}"
            )

            summary_resp = self.llm.invoke(
                [summary_prompt]
            )

            summary = summary_resp.content

        # Store query history
        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }


In [22]:
# Example usage
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = adv_rag.query(
    "what is attention mechanism",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

Retrieving documents for query: 'what is attention mechanism'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.36it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
(see Table A.4 for a complete reference).
Definition 2.1(Attention Mechanism).An attention mechanism is a differentiable functionA:R dq×
Rn×dk×Rn×dv→R dv that computes a weighted combination of values based on the compatibility between
a query and key

s.
Given:
•Query:q∈R dq
•Keys:K= (k 1,...,k n) wherek i∈R dk
•Values:V= (v 1,...,v n) wherev i∈R dv
The attention function computes:
ei = score(q,ki) fori= 1,...,n(2.8)
αi = exp(ei)Pn
j=1 exp(ej) (2.9)
A(q,K,V) =
nX
i=1
αivi (2.10)
where score :Rdq×Rdk→Ris a compatibility function, andα i are attention weights satisfyingPn
i=1αi =
1 andα i≥0.
The attention mechanism consists of three conceptual stages:
Stage 1 - Score Computation: For each positioni, compute a scalar scoree i measuring the com-
patibility or relevance between the queryqand keyk i. Different scoring functions are explored in the next
section.

18 CHAPTER 3. SELF-ATTENTION MECHANISMS
Example 3.2(Positional Patterns).Some attention heads learn to attend to specific relative positions.
For instance, a head might attend primarily to the next position (A i,i+1 ≈1) or the previous position
(Ai,i−1≈1). These patterns extract positional information.
Example 3.3(Syntactic Patterns).In language models, certain heads learn attenti

In [23]:
# Example usage
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = adv_rag.query(
    "what is CNN?",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

Retrieving documents for query: 'what is CNN?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Electronics 2021, 10, 2470 2 of 28
computer vision algorithms. A convolutional neural network is a method of deep learning
that takes an input image and assigns importance (learnable biases and weights) to various
objects in the image, distinguishing 

one from the other [1].
In comparison to other methods, CNN requires less preprocessing. Therefore, a CNN
is the most effective learning algorithm for comprehending picture material [1]. Further-
more, it has demonstrated exceptional image classiﬁcation, recognition, segmentation, and
retrieval [2]. The accomplishment of CNN has piqued the interest of people outside of
academia. Microsoft, Google, AT&T, NEC, and Facebook are among the companies engag-
ing in the development and advancement of CNN architecture [3]. In addition, they have
active research groups that are investigating novel CNN designs. At the moment, deep
CNN-based models are being used by the majority of front-runners in image processing

(Smarandache et al., 2019) reviewed trends in convolutional neural network architecture.
This paper primarily focuses on the design of the architecture of around 10 well-known
CNN models.
Similarly, there are many authors, such as Liu (2017), LeCun (2010), Guo (2016), and
Srinivas (201

### RAG Evaluation

We evaluate the pipeline in two layers:

**1. Retrieval quality** (is the right context being fetched?)
- **Hit Rate@k** - did at least one relevant chunk appear in the top-k?
- **MRR** - how high is the first relevant chunk ranked?
- **Precision@k** - what fraction of the top-k chunks are relevant?
- **Keyword Coverage** - what fraction of the expected key facts/terms appear in the retrieved context (a cheap proxy for context recall)

**2. Generation quality** (is the answer good, given the context?)
- **Faithfulness** (LLM-as-judge) - is every claim in the answer supported by the retrieved context? (hallucination check)
- **Answer Relevancy** (LLM-as-judge) - does the answer actually address the question?
- **Answer Similarity** - cosine similarity between the generated answer and a reference answer (embedding based)

A chunk counts as *relevant* if it comes from the expected PDF **and** contains at least one expected keyword.

In [24]:
# --- Evaluation dataset ---
# Edit / extend this with your own questions. Aim for 15-30 for stable numbers.
# expected_source : PDF file name that should be retrieved
# keywords        : terms that a relevant chunk should contain (case-insensitive)
# reference       : short ground-truth answer (used for Answer Similarity)

eval_dataset = [
    {
        "question": "What is the attention mechanism?",
        "expected_source": "Attention.pdf",
        "keywords": ["attention mechanism", "query", "keys", "values"],
        "reference": "An attention mechanism is a differentiable function that computes a weighted combination of values based on the compatibility between a query and keys."
    },
    {
        "question": "What is a convolutional neural network?",
        "expected_source": "CNN.pdf",
        "keywords": ["convolutional neural network", "input image", "deep learning"],
        "reference": "A CNN is a deep learning method that takes an input image and assigns learnable weights and biases to objects in the image to distinguish one from another."
    },
    {
        "question": "What is SadakVision?",
        "expected_source": "SadakVision \u2014 INDIS 2026.pdf",
        "keywords": ["sadakvision", "sensor", "vision"],
        "reference": "SadakVision is a sensor-and-vision fusion system where the sensor decides when the camera looks, and the vision model gives the verdict."
    },
    {
        "question": "How much compute does SadakVision save compared to vision-only?",
        "expected_source": "SadakVision \u2014 INDIS 2026.pdf",
        "keywords": ["89%", "frames", "4,620", "43,200"],
        "reference": "SadakVision uses about 89% less compute, sending 4,620 frames to the detector per hour versus 43,200."
    },
    {
        "question": "What did the randomized controlled trial find about teaching children to read in India?",
        "expected_source": "A better way to teach children to read in India.pdf",
        "keywords": ["randomized", "reading", "children", "Shishu"],
        "reference": "The randomized controlled trial evaluated a new method of teaching children to read in India (the Shishu program)."
    },
]
print(f"{len(eval_dataset)} eval questions loaded")

5 eval questions loaded


#### Retrieval metrics

In [25]:
import io, contextlib
import numpy as np

def is_relevant(doc, item):
    """A retrieved chunk is relevant if it's from the expected PDF and has >=1 expected keyword."""
    src = doc["metadata"].get("source_file", doc["metadata"].get("source", ""))
    if item["expected_source"].lower() not in src.lower():
        return False
    text = doc["content"].lower()
    return any(k.lower() in text for k in item["keywords"])


def dedupe(docs):
    """The vector store may contain duplicate chunks (see note below) - keep the first of each."""
    seen, out = set(), []
    for d in docs:
        key = d["content"].strip()
        if key not in seen:
            seen.add(key)
            out.append(d)
    return out


def evaluate_retrieval(dataset, retriever, k=5, dedupe_results=True):
    rows = []
    for item in dataset:
        # silence the retriever's verbose prints, fetch extra so dedupe can still fill k
        with contextlib.redirect_stdout(io.StringIO()):
            docs = retriever.retrieve(item["question"], top_k=k * 3 if dedupe_results else k)
        if dedupe_results:
            docs = dedupe(docs)
        docs = docs[:k]

        rel_flags = [is_relevant(d, item) for d in docs]
        first_rel = next((i for i, f in enumerate(rel_flags) if f), None)

        context = " ".join(d["content"].lower() for d in docs)
        kw_cov = np.mean([kw.lower() in context for kw in item["keywords"]])

        rows.append({
            "question": item["question"],
            "hit": float(any(rel_flags)),
            "mrr": 1.0 / (first_rel + 1) if first_rel is not None else 0.0,
            "precision": sum(rel_flags) / max(len(docs), 1),
            "keyword_coverage": float(kw_cov),
        })
    return rows

In [26]:
import pandas as pd

K = 5
retrieval_rows = evaluate_retrieval(eval_dataset, rag_retriever, k=K)
retrieval_df = pd.DataFrame(retrieval_rows)

display(retrieval_df.round(3))
print(f"\n===== Retrieval metrics (k={K}) =====")
print(f"Hit Rate@{K}       : {retrieval_df['hit'].mean():.3f}")
print(f"MRR               : {retrieval_df['mrr'].mean():.3f}")
print(f"Precision@{K}      : {retrieval_df['precision'].mean():.3f}")
print(f"Keyword Coverage  : {retrieval_df['keyword_coverage'].mean():.3f}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 53.26it/s]


,question,hit,mrr,precision,keyword_coverage
0,What is the attention mechanism?,1.0,1.0,0.6,1.000
1,What is a convolutional neural network?,1.0,1.0,0.8,1.000
2,What is SadakVision?,0.0,0.0,0.0,0.667
3,How much compute does SadakVision save compare...,1.0,1.0,0.5,1.000
4,What did the randomized controlled trial find ...,1.0,1.0,1.0,1.000



===== Retrieval metrics (k=5) =====
Hit Rate@5       : 0.800
MRR               : 0.800
Precision@5      : 0.580
Keyword Coverage  : 0.933


#### Generation metrics (LLM-as-judge + embedding similarity)

In [27]:
import json, re, time

def run_rag(question, retriever, llm, top_k=5):
    """Run the pipeline and return answer + the exact context the LLM saw."""
    with contextlib.redirect_stdout(io.StringIO()):
        docs = dedupe(retriever.retrieve(question, top_k=top_k * 3))[:top_k]
    if not docs:
        return {"answer": "No relevant context found.", "context": ""}
    context = "\n\n".join(d["content"] for d in docs)
    prompt = f"""Use the following context to answer the question concisely.
Only use information from the provided context.

Context:
{context}

Question: {question}

Answer:"""
    return {"answer": llm.invoke(prompt).content, "context": context}


def _parse_json(text):
    """Pull the first JSON object out of an LLM response."""
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0)) if m else {}


def judge_faithfulness(question, answer, context, llm):
    """Score 0-1: share of the answer's claims that are supported by the context."""
    prompt = f"""You are a strict evaluator. Break the ANSWER into individual factual claims, then
check each claim against the CONTEXT only (ignore your own knowledge).

CONTEXT:
{context}

ANSWER:
{answer}

Return ONLY JSON: {{"total_claims": <int>, "supported_claims": <int>, "reason": "<one sentence>"}}"""
    out = _parse_json(llm.invoke(prompt).content)
    total = out.get("total_claims", 0)
    return (out.get("supported_claims", 0) / total) if total else 0.0


def judge_answer_relevancy(question, answer, llm):
    """Score 0-1: does the answer directly address the question?"""
    prompt = f"""Rate how well the ANSWER addresses the QUESTION on a scale from 0 to 5
(0 = irrelevant / refuses, 3 = partially answers, 5 = fully and directly answers).
Do not judge factual correctness, only relevance and completeness.

QUESTION: {question}
ANSWER: {answer}

Return ONLY JSON: {{"score": <0-5>, "reason": "<one sentence>"}}"""
    out = _parse_json(llm.invoke(prompt).content)
    return float(out.get("score", 0)) / 5.0


def answer_similarity(answer, reference, embedding_manager):
    """Cosine similarity between generated answer and reference answer."""
    vecs = embedding_manager.model.encode([answer, reference], convert_to_numpy=True)
    return float(cosine_similarity([vecs[0]], [vecs[1]])[0][0])

In [28]:
def evaluate_generation(dataset, retriever, llm, embedding_manager, top_k=5, sleep_s=1.0):
    rows = []
    for item in dataset:
        try:
            res = run_rag(item["question"], retriever, llm, top_k=top_k)
            faith = judge_faithfulness(item["question"], res["answer"], res["context"], llm)
            relev = judge_answer_relevancy(item["question"], res["answer"], llm)
            sim = (answer_similarity(res["answer"], item["reference"], embedding_manager)
                   if item.get("reference") else np.nan)
            rows.append({
                "question": item["question"],
                "faithfulness": faith,
                "answer_relevancy": relev,
                "answer_similarity": sim,
                "answer": res["answer"],
            })
        except Exception as e:  # e.g. rate limit / bad JSON - keep going
            print(f"Skipped '{item['question']}': {e}")
        time.sleep(sleep_s)  # be gentle with Groq rate limits
    return rows

In [29]:
generation_rows = evaluate_generation(eval_dataset, rag_retriever, llm, embedding_manager, top_k=5)
generation_df = pd.DataFrame(generation_rows)

display(generation_df.drop(columns=["answer"]).round(3))
print("\n===== Generation metrics =====")
print(f"Faithfulness      : {generation_df['faithfulness'].mean():.3f}")
print(f"Answer Relevancy  : {generation_df['answer_relevancy'].mean():.3f}")
print(f"Answer Similarity : {generation_df['answer_similarity'].mean():.3f}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.65it/s]


,question,faithfulness,answer_relevancy,answer_similarity
0,What is the attention mechanism?,1.0,1.0,0.889
1,What is a convolutional neural network?,1.0,0.6,0.864
2,What is SadakVision?,1.0,1.0,0.758
3,How much compute does SadakVision save compare...,1.0,1.0,0.733
4,What did the randomized controlled trial find ...,0.0,1.0,0.776



===== Generation metrics =====
Faithfulness      : 0.800
Answer Relevancy  : 0.920
Answer Similarity : 0.804


#### Summary report

In [30]:
summary = pd.Series({
    f"Hit Rate@{K}": retrieval_df["hit"].mean(),
    "MRR": retrieval_df["mrr"].mean(),
    f"Precision@{K}": retrieval_df["precision"].mean(),
    "Keyword Coverage": retrieval_df["keyword_coverage"].mean(),
    "Faithfulness": generation_df["faithfulness"].mean(),
    "Answer Relevancy": generation_df["answer_relevancy"].mean(),
    "Answer Similarity": generation_df["answer_similarity"].mean(),
}, name="score").round(3)

display(summary.to_frame())

# Optional: save per-question results so you can compare runs after changing
# chunk_size, embedding model, top_k, prompt, etc.
retrieval_df.merge(generation_df.drop(columns=["answer"]), on="question").to_csv("../data/eval_results.csv", index=False)

,score
Hit Rate@5,0.800
MRR,0.800
Precision@5,0.580
Keyword Coverage,0.933
Faithfulness,0.800
Answer Relevancy,0.920
Answer Similarity,0.804


> **Note on duplicate chunks:** the earlier output shows the collection holding **2,275** documents although this PDF set only produces **440** chunks - the ingestion cell was run several times and `add_documents` generates a new random ID each run, so the same chunk is stored multiple times. Duplicates crowd the top-k with copies of one chunk and distort retrieval metrics. The eval above de-duplicates results to compensate, but the proper fix is to delete and re-create the collection (or use deterministic IDs, e.g. a hash of the chunk text, with `collection.upsert`).